# Predicting Tomorrow's Idle GPUs — v4 (견고성·원인 분석)

v3 결과: **test 주(140–146)** 에서는 튜닝한 LightGBM이 Baseline 1보다 같은 위험에서 더 많이 회수했지만,
**홀드아웃 5주(147–180)** 에서는 이득이 거의 없고 잘못 회수 건수는 오히려 늘었다.
v4는 이 차이가 어디서 오는지 가려낸다. 모든 실험은 **같은 6주(test + 홀드아웃 5주)를 같은 방식**으로 평가한다.

| 실험 | 질문 | 방법 |
| --- | --- | --- |
| E1 threshold 대 순위 | 모델의 **점수 순서**가 나쁜가, 아니면 직전 주에서 고른 **threshold가 어긋나는** 것인가? | 같은 점수에 정책 여러 개 적용: 직전 주 threshold / 같은 날 Baseline 1과 같은 개수 / 그 주에서 위험을 정확히 맞춘 사후 상한 |
| E2 seed | 이득이 학습 난수에 따라 흔들리는가? | LightGBM·XGBoost 튜닝 설정을 seed 5개로 |
| E3 변수 묶음 | Baseline 1(오늘 idle_ratio)에 무엇을 더해야 이득이 생기나? | 변수 묶음을 차례로 더하기 / 하나씩 빼기 (LightGBM) |
| E4 학습 기간 | 오래된 날이 오히려 방해되나(분포 변화)? | 전체 과거 / 최근 56일 / 최근 28일로 학습 |

평가 방식 (v3 홀드아웃과 같음): 주 `[a, b]`를 예측할 때 최종 모델은 `day ≤ a−2`로 학습,
threshold는 `day ≤ a−9` 모델을 `a−8 … a−2`에 적용해서 고른다. 하이퍼파라미터는 v3에서 고른 값으로 고정.

**정책** (모두 같은 점수에 적용)
- `F1 최대 (직전 주)`, `위험 맞춤 GPU-h (직전 주)`, `위험 맞춤 건수 (직전 주)`: v3와 같음
- `같은 날 B1과 같은 개수`: 그날 Baseline 1이 회수하는 app 수 k만큼 점수 상위 k개를 회수 — 정답 없이 **그날 바로 쓸 수 있는** 규칙.
  threshold 이동 문제를 없애고 "누구를 고르느냐"만 비교한다
- `사후 맞춤 GPU-h (상한)`, `사후 맞춤 건수 (상한)`: 그 주의 정답을 보고 Baseline 1과 잘못 회수량을 정확히 맞춘 threshold —
  운영에서는 불가능한 **순위 품질의 상한**

## 0. 설정 (v3와 같음)

In [1]:
# CUDA 라이브러리(cupy/cuml/xgboost)를 import하기 전에 실행해야 한다.
import os
import subprocess
import time


def gpu_status():
    out = subprocess.run(["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used",
                          "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
    rows = []
    for line in out.splitlines():
        parts = [p.strip() for p in line.split(",")]
        if len(parts) == 3 and all(p.isdigit() for p in parts):
            rows.append(tuple(map(int, parts)))
    return rows


def spare_gpus(n_samples=6, max_util=30, max_mem_mb=4000):
    # 잠깐 동안 여러 번 재서 최대 사용률·메모리가 기준 이하인 GPU만 고른다 (다른 사람이 쓰는 GPU 제외)
    peak = {}
    for _ in range(n_samples):
        for i, u, m in gpu_status():
            pu, pm = peak.get(i, (0, 0))
            peak[i] = (max(pu, u), max(pm, m))
        time.sleep(0.5)
    return [i for i, (u, m) in sorted(peak.items()) if u <= max_util and m <= max_mem_mb], peak


GPUS, peak = spare_gpus()
if os.environ.get("GPU_IDS"):
    GPUS = [int(x) for x in os.environ["GPU_IDS"].split(",")]
print("GPU 상태 (최대 사용률 %, 메모리 MiB):", peak)
assert GPUS, "남는 GPU가 없음"
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(map(str, GPUS))
N_GPU = len(GPUS)
DEVICES = list(range(N_GPU))   # CUDA_VISIBLE_DEVICES 안에서의 번호
print(f"사용할 GPU: 물리 번호 {GPUS} -> 장치 {DEVICES}")

GPU 상태 (최대 사용률 %, 메모리 MiB): {0: (28, 1415), 1: (0, 5)}
사용할 GPU: 물리 번호 [0, 1] -> 장치 [0, 1]


In [2]:
import json
import queue
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

ROOT = Path.cwd().resolve().parent          # notebooks/ 의 상위 = 프로젝트 루트
OUT = ROOT / "outputs" / "v4"
V3_OUT = ROOT / "outputs" / "v3"
OUT.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 0

# 실행 규모 (시간이 부족하면 줄인다)
N_TRIALS = 40            # Optuna 시도 횟수 (모델당)
OPTUNA_TIMEOUT = 2400    # 모델당 최대 초
N_BOOT = 2000            # 부트스트랩 반복
SVM_TRAIN_N = None       # cuML SVC 학습 행 수 (None = train 전체; GPU에서 39만 행도 수 초)
SMOKE = os.environ.get("V4_SMOKE") == "1"   # 코드 점검용 축소 실행
if SMOKE:
    N_TRIALS, N_BOOT = 2, 50

import cupy as cp
import xgboost as xgb
import lightgbm as lgb
import optuna
from sklearn.metrics import average_precision_score

optuna.logging.set_verbosity(optuna.logging.WARNING)
print("cupy", cp.__version__, "| xgboost", xgb.__version__, "| lightgbm", lgb.__version__, "| optuna", optuna.__version__)
for d in DEVICES:
    print(f"  장치 {d}:", cp.cuda.runtime.getDeviceProperties(d)["name"].decode())

/ssd/jylee/gpu-reclaim/env-gpu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cupy 12.2.0 | xgboost 2.1.4 | lightgbm 4.6.0 | optuna 5.1.0
  장치 0: NVIDIA RTX A4000
  장치 1: NVIDIA RTX A4000


## 1. 데이터 (v3와 같음)

In [3]:
DATA = ROOT / "data" / "processed" / "app_day_v2_max181.parquet"
if not DATA.exists():   # 없으면 v2 전처리를 다시 실행 (CPU, 약 7분)
    subprocess.run([sys.executable, "-I", str(ROOT / "scripts" / "build_app_day_v2.py"), "181"], check=True)

df = pd.read_parquet(DATA)
df["day_of_week"] = pd.Categorical(df["day_of_week"], categories=range(7))   # parquet 왕복에서 int로 풀림 -> 복원
assert "app_id" in df.columns

SPLITS = {"train": (0, 132), "val": (133, 139), "test": (140, 146), "holdout": (147, 180)}


def in_days(data, days):
    return data["day"].between(*days)


def part(name, data=None):
    data = df if data is None else data
    return data[in_days(data, SPLITS[name])]


sizes = {k: len(part(k)) for k in SPLITS}
print(f"전체 {len(df):,} 행 | " + " | ".join(f"{k} {v:,}" for k, v in sizes.items()))
# v2와 같은 데이터인지 확인 (v2: train+val 414,663 / test 21,342)
assert sizes["train"] + sizes["val"] == 414_663 and sizes["test"] == 21_342
print("idle(1) 비율:", {k: round(part(k)["is_mostly_idle_next"].mean() * 100, 2) for k in SPLITS})

전체 535,846 행 | train 393,828 | val 20,835 | test 21,342 | holdout 99,841
idle(1) 비율: {'train': 36.44, 'val': 32.48, 'test': 32.55, 'holdout': 34.14}


## 2. 인코더·공통 함수·Baseline·학습 함수 (v3와 같음)

In [4]:
# 모델별 인코딩 파이프라인
# 1.2 전처리와 달리 인코더는 "어떤 범주가 있는지"를 데이터에서 학습(fit)한다.
# -> 반드시 train으로만 fit하고 val / test에는 transform만 한다. (모델 파이프라인 안에 넣어서 사용)
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

TARGET_COLS = ["is_mostly_idle_next"]
AUX_COLS = ["day", "gpu_hours_next", "is_mostly_idle", "app_id"]  # v3: app_id 추가 (부트스트랩·시뮬레이션용)   # 모델 input에서 제외 (오늘 idle_ratio는 input으로 사용)


def split_xyw(data, target="is_mostly_idle_next"):
    """preprocessed_df -> (X, y, w). X에는 타겟과 보조 변수가 들어가지 않는다.

    - y: is_mostly_idle_next (다음 날 사실상 놀기만 할지, 0/1)
    - w: gpu_hours_next (sample_weight) - 다음 날 회수할 수 있는 GPU 양
    """
    X = data.drop(columns=TARGET_COLS + AUX_COLS)
    return X, data[target], data["gpu_hours_next"].copy()


class ColumnsByDtype:
    """ColumnTransformer용 컬럼 선택기: dtype("category" / "number")으로 고르고 exclude는 뺀다.

    - exclude: 뺄 컬럼 이름
    - exclude_prefix: 이 접두어로 시작하는 컬럼도 뺀다 (예: "outlier_")
    (속성 이름을 dtype으로 하면 sklearn이 배열로 오인하므로 include를 쓴다)
    """

    def __init__(self, include, exclude=(), exclude_prefix=()):
        self.include = include
        self.exclude = tuple(exclude)
        self.exclude_prefix = tuple(exclude_prefix)

    def __call__(self, X):
        return [c for c in X.select_dtypes(include=self.include).columns
                if c not in self.exclude and not c.startswith(self.exclude_prefix)]


class ColumnsByPrefix:
    """ColumnTransformer용 컬럼 선택기: 이름이 prefix로 시작하는 컬럼을 고른다."""

    def __init__(self, prefix):
        self.prefix = prefix

    def __call__(self, X):
        return [c for c in X.columns if c.startswith(self.prefix)]


# ---------------------------------------------------------------------------
# Logistic regression / SVM용
# ---------------------------------------------------------------------------
def build_linear_encoder(min_frequency=0.01, exclude=("site_id",), flag_prefix="outlier_",
                         log_cols=("gpu_hours", "gpu_hours_per_gpu")):
    """범주형 one-hot + 수치형 결측 대체·표준화 + 플래그는 0/1 그대로.

    - 범주형: OneHotEncoder
        * min_frequency: 전체의 1% 미만인 드문 범주는 "infrequent" 컬럼 하나로 묶음
        * handle_unknown: train에 없던 범주가 val / test에 나오면 infrequent로 처리 (에러 방지)
        * NaN은 하나의 범주로 취급
    - 수치형: 중앙값으로 결측 대체 -> StandardScaler (선형 모델은 NaN과 스케일 차이에 민감)
    - log_cols(gpu_hours, gpu_hours_per_gpu): 중앙값 대체 -> log1p -> StandardScaler
        (0.03 ~ 수만까지 치우친 분포라 표준화만으로는 극단값에 계수가 끌려감)
    - 플래그(outlier_*): 0/1 그대로 통과 (드문 0/1 값을 표준화하면 1이 7~19 같은 큰 값이 되므로)
    - exclude: cluster_id와 거의 1:1인 site_id는 중복 정보라 기본으로 제외
    """
    categorical = OneHotEncoder(min_frequency=min_frequency, handle_unknown="infrequent_if_exist")
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    log_numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", StandardScaler()),
    ])
    return ColumnTransformer([
        ("cat", categorical, ColumnsByDtype("category", exclude)),
        ("num", numeric, ColumnsByDtype("number", tuple(exclude) + tuple(log_cols),
                                        exclude_prefix=(flag_prefix,))),
        ("log", log_numeric, list(log_cols)),
        ("flag", "passthrough", ColumnsByPrefix(flag_prefix)),
    ], remainder="drop")


# ---------------------------------------------------------------------------
# LightGBM / XGBoost용
# ---------------------------------------------------------------------------
class CategoryAligner(BaseEstimator, TransformerMixin):
    """one-hot 없이 category dtype 그대로 넘긴다 (트리 모델이 범주형을 직접 처리).

    fit 때 train에 실제로 등장한 범주만 기억하고, transform 때 범주 목록을 그것으로 맞춘다.
    -> train / val / test의 범주 코드가 항상 같고, train에 없던 범주는 NaN(결측)으로 처리된다.
    수치형 NaN은 트리 모델이 직접 처리하므로 건드리지 않는다.
    """

    def __init__(self, exclude=()):
        self.exclude = exclude

    def fit(self, X, y=None):
        cols = ColumnsByDtype("category", self.exclude)(X)
        self.categories_ = {c: X[c].cat.remove_unused_categories().cat.categories for c in cols}
        return self

    def transform(self, X):
        X = X.drop(columns=[c for c in self.exclude if c in X.columns]).copy()
        for c, cats in self.categories_.items():
            X[c] = X[c].cat.set_categories(cats)
        return X


def build_tree_encoder(exclude=()):
    """트리 모델은 cluster_id와 site_id의 중복에 크게 영향받지 않으므로 기본으로 둘 다 사용."""
    return CategoryAligner(exclude=exclude)


# ---------------------------------------------------------------------------
# 사용 예 (모델 셀에서)
#   X_train, y_train, w_train = split_xyw(train_df)
#   model = Pipeline([("encode", build_linear_encoder()),
#                     ("model", LogisticRegression(max_iter=1000))])
#   model.fit(X_train, y_train, model__sample_weight=w_train)
# ---------------------------------------------------------------------------

In [5]:
# v2 XGBoost 셀의 단조 제약 정의 (v3에서는 쓸지 말지를 CV로 정한다)
XGB_MONOTONE = {
    "idle_ratio": 1,        # 오늘 많이 놀수록 내일도 놀 가능성 ↑
    "idle_ratio_lag1": 1,   # 어제 많이 놀수록 ↑
    "idle_streak": 1,       # 오래 연속으로 놀수록 ↑
    "sm_util": -1,          # GPU 연산 사용률이 높을수록 ↓
}

LINEAR_FEATURES = None


def norm_w(w):
    # 학습용 가중치: 평균 1로 정규화 (gpu_hours_next 값이 수천까지 커서 min_child_weight 등의 의미가 흔들리지 않게)
    w = np.asarray(w, dtype="float64")
    return w / w.mean()


def dense32(a):
    import scipy.sparse as sp
    return (a.toarray() if sp.issparse(a) else np.asarray(a)).astype("float32")


# ---------------------------------------------------------------------------
# 평가: 점수(score)와 0/1 판정(pred)을 받아 GPU-hours 기준 + 건수 기준 지표를 낸다
# ---------------------------------------------------------------------------
def evaluate(y, score, pred, w):
    y = np.asarray(y).astype(int); pred = np.asarray(pred).astype(int); w = np.asarray(w, dtype="float64")
    tp, fp, fn = (pred == 1) & (y == 1), (pred == 1) & (y == 0), (pred == 0) & (y == 1)
    TP, FP, FN = w[tp].sum(), w[fp].sum(), w[fn].sum()
    return {
        "AP": average_precision_score(y, score, sample_weight=w),
        "F1": 2 * TP / max(2 * TP + FP + FN, 1e-12),
        "precision": TP / max(TP + FP, 1e-12),
        "recall": TP / max(TP + FN, 1e-12),
        "회수 GPU-h": TP, "잘못 회수 GPU-h": FP, "놓친 idle GPU-h": FN,
        "잘못 회수 건수": int(fp.sum()), "회수 건수": int((pred == 1).sum()),
    }


def threshold_curve(y, score, w):
    # 점수 내림차순으로 threshold를 낮춰 가며 누적 회수/잘못 회수 (같은 점수는 한 번에)
    y = np.asarray(y).astype(int); w = np.asarray(w, dtype="float64"); s = np.asarray(score, dtype="float64")
    o = np.argsort(-s, kind="stable"); s, y, w = s[o], y[o], w[o]
    tp, fp, fpn = np.cumsum(w * y), np.cumsum(w * (1 - y)), np.cumsum(1 - y)
    last = np.r_[s[1:] != s[:-1], True]
    P = (w * y).sum()
    c = pd.DataFrame({"thr": s[last], "회수 GPU-h": tp[last], "잘못 회수 GPU-h": fp[last], "잘못 회수 건수": fpn[last]})
    c["F1"] = 2 * c["회수 GPU-h"] / (c["회수 GPU-h"] + c["잘못 회수 GPU-h"] + P)
    return c


POLICIES = ["F1 최대", "위험 맞춤 (GPU-h)", "위험 맞춤 (건수)"]


def pick_threshold(y, score, w, policy, budget=None):
    # val에서 threshold를 고른다.
    #  - F1 최대: v2와 같은 기준 (gpu_hours_next 가중 F1)
    #  - 위험 맞춤: 잘못 회수 비율이 budget(= 같은 기간 Baseline 1의 비율) 이하인 threshold 중 회수량 최대
    #    GPU-h: 잘못 회수 GPU-h / 전체 다음날 GPU-h,  건수: 잘못 회수 건수 / 전체 행 수
    c = threshold_curve(y, score, w)
    if policy == "F1 최대":
        return float(c.loc[c["F1"].idxmax(), "thr"])
    if policy == "위험 맞춤 (GPU-h)":
        ok = c["잘못 회수 GPU-h"] / np.sum(w) <= budget
    else:
        ok = c["잘못 회수 건수"] / len(y) <= budget
    if not ok.any():
        return float("inf")
    return float(c.loc[ok[ok].index[-1], "thr"])   # 조건을 만족하는 가장 낮은 threshold = 회수량 최대


def risk_budget(y, base_pred, w):
    # Baseline 1의 잘못 회수 비율 (GPU-h, 건수)
    y = np.asarray(y).astype(int); p = np.asarray(base_pred).astype(int); w = np.asarray(w, dtype="float64")
    fp = (p == 1) & (y == 0)
    return {"위험 맞춤 (GPU-h)": w[fp].sum() / w.sum(), "위험 맞춤 (건수)": fp.sum() / len(y)}

In [6]:
def base1(data):
    return data["idle_ratio"].to_numpy(dtype="float64"), data["is_mostly_idle"].to_numpy().astype(int)


def base2(data, idle_rate):
    util = data["sm_util"].to_numpy(dtype="float64")
    util = np.where(np.isnan(util), np.nanmax(util) + 1, util)
    n = int(round(idle_rate * len(data)))
    pred = np.zeros(len(data), dtype=int)
    pred[np.argsort(util, kind="stable")[:n]] = 1
    return -util, pred, n

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


class Fitted:
    # 인코더 + 모델을 묶어서 score(data) -> 내일 idle 점수
    def __init__(self, enc, model, kind, best_iter=None):
        self.enc, self.model, self.kind, self.best_iter = enc, model, kind, best_iter

    def score(self, data):
        X, _, _ = split_xyw(data)
        Z = self.enc.transform(X)
        if self.kind == "svm":
            return np.asarray(self.model.decision_function(dense32(Z))).astype("float64")
        if self.kind == "rf":
            return np.asarray(self.model.predict_proba(dense32(Z)))[:, 1].astype("float64")
        return self.model.predict_proba(Z)[:, 1].astype("float64")


def fit_model(kind, params, tr, device=0, ev=None):
    # kind: lr / svm / rf / lgbm / xgb,  ev가 있으면 트리 모델은 early stopping에 사용
    X, y, w = split_xyw(tr)
    p = dict(params)
    if kind == "lr":
        enc = build_linear_encoder().fit(X)
        m = LogisticRegression(max_iter=2000, C=p["C"]).fit(enc.transform(X), y, sample_weight=norm_w(w))
        return Fitted(enc, m, kind)
    if kind == "svm":
        from cuml.svm import SVC as cuSVC
        enc = build_linear_encoder().fit(X)
        n = len(X) if SVM_TRAIN_N is None else min(SVM_TRAIN_N, len(X))
        idx = np.random.default_rng(RANDOM_STATE).choice(len(X), size=n, replace=False) if n < len(X) else np.arange(len(X))
        Z = dense32(enc.transform(X.iloc[idx]))
        with cp.cuda.Device(device):
            m = cuSVC(C=p["C"], gamma=p["gamma"], cache_size=4096)
            m.fit(Z, y.to_numpy()[idx].astype("float32"), sample_weight=norm_w(w.to_numpy()[idx]))
        return Fitted(enc, m, kind)
    if kind == "rf":
        from sklearn.ensemble import RandomForestClassifier
        enc = build_linear_encoder().fit(X)
        m = RandomForestClassifier(n_estimators=p.get("n_estimators", 300), max_depth=p["max_depth"],
                                   max_features=p["max_features"], n_jobs=-1, random_state=RANDOM_STATE)
        m.fit(dense32(enc.transform(X)), y, sample_weight=norm_w(w))
        return Fitted(enc, m, kind)

    enc = build_tree_encoder().fit(X)
    Z = enc.transform(X)
    n_est = p.pop("n_estimators", 3000)
    if ev is not None:
        Xe, ye, we = split_xyw(ev)
        Ze = enc.transform(Xe)
    if kind == "lgbm":
        m = lgb.LGBMClassifier(device_type="cuda", gpu_device_id=device, n_estimators=n_est,
                               random_state=RANDOM_STATE, verbose=-1, bagging_freq=1, **p)
        kw = {}
        if ev is not None:
            kw = dict(eval_set=[(Ze, ye)], eval_sample_weight=[norm_w(we)], eval_metric="average_precision",
                      callbacks=[lgb.early_stopping(100, first_metric_only=True, verbose=False)])
        m.fit(Z, y, sample_weight=norm_w(w), **kw)
        bi = m.best_iteration_ if ev is not None else n_est
        return Fitted(enc, m, kind, bi)
    if kind == "xgb":
        mono = p.pop("monotone", False)
        m = xgb.XGBClassifier(enable_categorical=True, tree_method="hist", device=f"cuda:{device}",
                              n_estimators=n_est, random_state=RANDOM_STATE, eval_metric="aucpr",
                              monotone_constraints=XGB_MONOTONE if mono else None,
                              early_stopping_rounds=100 if ev is not None else None, **p)
        kw = dict(eval_set=[(Ze, ye)], sample_weight_eval_set=[norm_w(we)]) if ev is not None else {}
        m.fit(Z, y, sample_weight=norm_w(w), verbose=False, **kw)
        bi = (m.best_iteration + 1) if ev is not None else n_est
        return Fitted(enc, m, kind, bi)
    raise ValueError(kind)


def wap(data, score):
    _, y, w = split_xyw(data)
    return average_precision_score(y, score, sample_weight=w)

In [8]:
def boot_diff(data, preds, base_pred, n_boot=N_BOOT, seed=RANDOM_STATE):
    _, y, w = split_xyw(data)
    y = y.to_numpy().astype(int); w = w.to_numpy().astype("float64")
    codes, uniq = pd.factorize(data["app_id"].astype(str))
    n_app = len(uniq)

    def per_app(pred):
        pred = np.asarray(pred).astype(int)
        cols = np.stack([w * (pred == 1) * (y == 1), w * (pred == 1) * (y == 0), w * (pred == 0) * (y == 1),
                         ((pred == 1) & (y == 0)).astype(float)], axis=1)
        M = np.zeros((n_app, 4)); np.add.at(M, codes, cols); return M

    rng = np.random.default_rng(seed)
    # 각 반복에서 app별 뽑힌 횟수 (다항분포) -> 행렬곱 한 번으로 합계 (GPU)
    C = cp.asarray(rng.multinomial(n_app, np.full(n_app, 1 / n_app), size=n_boot).astype("float32"))
    MB = cp.asarray(per_app(base_pred), dtype=cp.float32)
    SB = cp.asnumpy(C @ MB)
    out = []
    for name, pred in preds.items():
        S = cp.asnumpy(C @ cp.asarray(per_app(pred), dtype=cp.float32))
        f1 = lambda T: 2 * T[:, 0] / (2 * T[:, 0] + T[:, 1] + T[:, 2])
        d = {"Δ회수 GPU-h": S[:, 0] - SB[:, 0], "Δ잘못 회수 GPU-h": S[:, 1] - SB[:, 1],
             "Δ잘못 회수 건수": S[:, 3] - SB[:, 3], "ΔF1": f1(S) - f1(SB)}
        r = {"방법": name}
        for k, v in d.items():
            lo, hi = np.percentile(v, [2.5, 97.5])
            r[k] = f"{np.mean(v):+,.4g} [{lo:+,.4g}, {hi:+,.4g}]"
        out.append(r)
    return pd.DataFrame(out)

## 3. 공통 평가 틀

- `EVAL_WEEKS`: test 주 + 홀드아웃 5주. 합계는 **6주 전체**와 **홀드아웃 5주** 두 가지로 낸다
- 하이퍼파라미터: v3의 `best_params.json`
- 점수는 전부 `outputs/v4/scores.parquet`에 저장 (나중에 다시 학습하지 않고 다른 정책을 시험할 수 있게)

In [9]:
BEST = json.load(open(V3_OUT / "best_params.json"))
print("v3에서 고른 하이퍼파라미터:", BEST)
HOLD_WEEKS = [(147, 153), (154, 160), (161, 167), (168, 174), (175, 180)]
EVAL_WEEKS = [(140, 146)] + HOLD_WEEKS
if SMOKE:
    EVAL_WEEKS = EVAL_WEEKS[:2]
WEEK_NAMES = [f"{a}–{b}" for a, b in EVAL_WEEKS]
POOLS = {"6주 (140–180)": WEEK_NAMES, "홀드아웃 5주 (147–180)": [n for n in WEEK_NAMES if n != "140–146"]}

# 주별 고정 정보 (정답·가중치·Baseline 1) — 변수 묶음 실험에서도 원본 df 기준
WK = []
for a, b in EVAL_WEEKS:
    wk, cal = df[df["day"].between(a, b)], df[df["day"].between(a - 8, a - 2)]
    _, yk, wk_w = split_xyw(wk)
    _, yc, wc = split_xyw(cal)
    WK.append({"name": f"{a}–{b}", "a": a, "b": b, "wk": wk, "cal": cal,
               "yk": yk.to_numpy().astype(int), "wk_w": wk_w.to_numpy().astype("float64"),
               "yc": yc.to_numpy().astype(int), "wc": wc.to_numpy().astype("float64"),
               "b1k": base1(wk)[1], "b1c": base1(cal)[1], "s1k": base1(wk)[0], "s1c": base1(cal)[0],
               "day": wk["day"].to_numpy(), "workload": wk["workload"].astype(str).to_numpy()})
    WK[-1]["bud"] = risk_budget(WK[-1]["yc"], WK[-1]["b1c"], WK[-1]["wc"])
print("평가 주:", [(w["name"], len(w["wk"])) for w in WK])


def topk_per_day(s, day, base):
    # 그날 Baseline 1이 회수하는 개수 k만큼 점수 상위 k개 (정답 없이 그날 바로 쓸 수 있음)
    pred = np.zeros(len(s), dtype=bool)
    for d in np.unique(day):
        idx = np.flatnonzero(day == d)
        k = int(base[idx].sum())
        if k:
            pred[idx[np.argsort(-s[idx], kind="stable")[:k]]] = True
    return pred


def oracle_pred(y, s, w, base, col):
    # 그 주 정답을 보고 Baseline 1과 같은 잘못 회수량(col) 이하에서 회수 최대인 threshold (상한, 운영 불가)
    c = threshold_curve(y, s, w)
    fp = (np.asarray(base) == 1) & (y == 0)
    b = w[fp].sum() if col == "잘못 회수 GPU-h" else fp.sum()
    ok = c[col] <= b + 1e-9
    thr = float(c.loc[ok[ok].index[-1], "thr"]) if ok.any() else np.inf
    return s >= thr


POL_NAMES = ["F1 최대 (직전 주)", "위험 맞춤 GPU-h (직전 주)", "위험 맞춤 건수 (직전 주)",
             "같은 날 B1과 같은 개수", "사후 맞춤 GPU-h (상한)", "사후 맞춤 건수 (상한)"]


def policy_preds(W, s_c, s_k):
    yc, wc, bud = W["yc"], W["wc"], W["bud"]
    return {
        "F1 최대 (직전 주)": s_k >= pick_threshold(yc, s_c, wc, "F1 최대"),
        "위험 맞춤 GPU-h (직전 주)": s_k >= pick_threshold(yc, s_c, wc, "위험 맞춤 (GPU-h)", bud["위험 맞춤 (GPU-h)"]),
        "위험 맞춤 건수 (직전 주)": s_k >= pick_threshold(yc, s_c, wc, "위험 맞춤 (건수)", bud["위험 맞춤 (건수)"]),
        "같은 날 B1과 같은 개수": topk_per_day(s_k, W["day"], W["b1k"]),
        "사후 맞춤 GPU-h (상한)": oracle_pred(W["yk"], s_k, W["wk_w"], W["b1k"], "잘못 회수 GPU-h"),
        "사후 맞춤 건수 (상한)": oracle_pred(W["yk"], s_k, W["wk_w"], W["b1k"], "잘못 회수 건수"),
    }


WEEKLY, POOLED, PRED_STORE, SCORE_STORE = [], [], {}, {}


def record(exp, setting, method, seed, week_scores=None, fixed_pred=None):
    # week_scores: 주별 (cal 점수, week 점수) 목록. fixed_pred: 정책 없는 0/1 판정(Baseline 1)
    preds_by_pol = {}
    for i, W in enumerate(WK):
        if fixed_pred is not None:
            pp, sk = {"-": fixed_pred[i]}, W["s1k"]
        else:
            s_c, sk = week_scores[i]
            pp = policy_preds(W, s_c, sk)
            SCORE_STORE[(exp, setting, method, seed, W["name"])] = sk
        for pol, pred in pp.items():
            WEEKLY.append({"실험": exp, "설정": setting, "방법": method, "seed": seed, "주": W["name"], "정책": pol,
                           **evaluate(W["yk"], sk, pred, W["wk_w"])})
            preds_by_pol.setdefault(pol, []).append(np.asarray(pred).astype(int))
    for pol, plist in preds_by_pol.items():
        PRED_STORE[(exp, setting, method, seed, pol)] = plist
        for pool, names in POOLS.items():
            idx = [i for i, W in enumerate(WK) if W["name"] in names]
            if not idx:
                continue
            y = np.concatenate([WK[i]["yk"] for i in idx]); w = np.concatenate([WK[i]["wk_w"] for i in idx])
            p = np.concatenate([plist[i] for i in idx]); wl = np.concatenate([WK[i]["workload"] for i in idx])
            for seg in ["전체", "online", "offline"]:
                m = np.ones(len(y), bool) if seg == "전체" else (wl == seg)
                r = evaluate(y[m], p[m].astype(float), p[m], w[m]); r.pop("AP")
                POOLED.append({"실험": exp, "설정": setting, "방법": method, "seed": seed, "구간": pool, "workload": seg,
                               "정책": pol, **r})


def walk_forward(kind, params, data=None, seed=0, window=None):
    # 주마다 (cal 점수, week 점수). window: 최근 며칠만 학습 (None = 전체 과거)
    global RANDOM_STATE
    data = df if data is None else data
    RANDOM_STATE = seed
    out = []
    try:
        for W in WK:
            a = W["a"]

            def trn(end):
                m = data["day"] <= end
                if window:
                    m &= data["day"] > end - window
                return data[m]

            cal, wk = data[data["day"].between(a - 8, a - 2)], data[data["day"].between(a, W["b"])]   # WK와 같은 순서
            s_c = fit_model(kind, params, trn(a - 9), device=DEVICES[0]).score(cal)
            s_k = fit_model(kind, params, trn(a - 2), device=DEVICES[0]).score(wk)
            out.append((s_c, s_k))
    finally:
        RANDOM_STATE = 0
    return out


def pooled_view(exp, pol_list=None, seg="전체"):
    p = pd.DataFrame(POOLED)
    p = p[(p["실험"] == exp) & (p["workload"] == seg)]
    b = pd.DataFrame(POOLED)
    b = b[(b["실험"] == "E1") & (b["방법"] == "Baseline 1: 오늘 idle") & (b["workload"] == seg)].set_index("구간")
    for c in ["회수 GPU-h", "잘못 회수 GPU-h", "잘못 회수 건수", "F1"]:
        p[f"Δ{c} vs B1"] = p[c] - p["구간"].map(b[c])
    if pol_list:
        p = p[p["정책"].isin(pol_list + ["-"])]
    return p


T0 = time.time()
def elapsed():
    return f"{(time.time() - T0) / 60:.1f}분"

v3에서 고른 하이퍼파라미터: {'xgb': {'learning_rate': 0.038975798381148376, 'max_depth': 7, 'min_child_weight': 1.2174277255542338, 'subsample': 0.7358009917978997, 'colsample_bytree': 0.6614737622332093, 'reg_lambda': 0.0028935189508194703, 'monotone': False, 'n_estimators': 87}, 'lgbm': {'learning_rate': 0.022520188368138162, 'num_leaves': 221, 'min_child_samples': 52, 'subsample': 0.8400785647649014, 'colsample_bytree': 0.5963512927119154, 'reg_lambda': 0.09307568252326939, 'n_estimators': 268}, 'lr': {'C': 0.01}, 'svm': {'C': 0.1, 'gamma': 0.01}, 'rf': {'max_depth': 16, 'max_features': 0.5}}
평가 주: [('140–146', 21342), ('147–153', 21800), ('154–160', 22036), ('161–167', 21818), ('168–174', 15446), ('175–180', 18741)]


## 4. E1 — threshold 문제인가, 순위 문제인가

방법: Baseline 1, Baseline 1′(idle_ratio 점수에 정책 적용), LR, RF, LightGBM 튜닝, XGBoost 튜닝, 두 부스팅의 확률 평균. seed 0.
- `사후 맞춤`에서도 모델이 Baseline 1과 비슷하면 → **점수 순서 자체**가 Baseline 1보다 낫지 않다
- `사후 맞춤`에서는 낫고 `직전 주`에서만 못하면 → **threshold가 주마다 어긋나는** 문제
- `같은 날 B1과 같은 개수`는 threshold 없이 바로 쓸 수 있는 대안

In [10]:
record("E1", "기본", "Baseline 1: 오늘 idle", 0, fixed_pred=[W["b1k"] for W in WK])
record("E1", "기본", "Baseline 1′ (idle_ratio 점수)", 0, week_scores=[(W["s1c"], W["s1k"]) for W in WK])
E1_MODELS = {"Logistic Regression": ("lr", BEST["lr"]), "Random Forest": ("rf", BEST["rf"]),
             "LightGBM 튜닝": ("lgbm", BEST["lgbm"]), "XGBoost 튜닝": ("xgb", BEST["xgb"])}
E1_SCORES = {}
for name, (kind, p) in E1_MODELS.items():
    t0 = time.time()
    E1_SCORES[name] = walk_forward(kind, p)
    record("E1", "기본", name, 0, week_scores=E1_SCORES[name])
    print(f"E1 {name}: {time.time() - t0:.0f}초 (누적 {elapsed()})", flush=True)
ens = [((a1 + a2) / 2, (b1 + b2) / 2) for (a1, b1), (a2, b2) in zip(E1_SCORES["LightGBM 튜닝"], E1_SCORES["XGBoost 튜닝"])]
record("E1", "기본", "LGBM+XGB 평균", 0, week_scores=ens)

e1 = pooled_view("E1")
e1.to_csv(OUT / "e1_pooled.csv", index=False)
pd.DataFrame([r for r in WEEKLY if r["실험"] == "E1"]).to_csv(OUT / "e1_weekly.csv", index=False)
cols = ["방법", "정책", "F1", "회수 GPU-h", "잘못 회수 GPU-h", "잘못 회수 건수", "ΔF1 vs B1", "Δ회수 GPU-h vs B1",
        "Δ잘못 회수 GPU-h vs B1", "Δ잘못 회수 건수 vs B1"]
for pool in POOLS:
    print(f"\n=== E1 | {pool} | workload 전체 ===")
    display(e1[e1["구간"] == pool][cols].round(4))

E1 Logistic Regression: 89초 (누적 1.5분)


E1 Random Forest: 1245초 (누적 22.2분)


E1 LightGBM 튜닝: 192초 (누적 25.5분)


E1 XGBoost 튜닝: 30초 (누적 25.9분)



=== E1 | 6주 (140–180) | workload 전체 ===


,방법,정책,F1,회수 GPU-h,잘못 회수 GPU-h,잘못 회수 건수,ΔF1 vs B1,Δ회수 GPU-h vs B1,Δ잘못 회수 GPU-h vs B1,Δ잘못 회수 건수 vs B1
0,Baseline 1: 오늘 idle,-,0.8738,4.075875e+06,713751.7830,1795,0.0000,0.0000,0.0000,0
6,Baseline 1′ (idle_ratio 점수),F1 최대 (직전 주),0.8703,4.047490e+06,714936.4777,1756,-0.0035,-28385.1892,1184.6947,-39
12,Baseline 1′ (idle_ratio 점수),위험 맞춤 GPU-h (직전 주),0.8738,4.076237e+06,714496.1664,1798,-0.0000,361.6167,744.3834,3
18,Baseline 1′ (idle_ratio 점수),위험 맞춤 건수 (직전 주),0.8738,4.076237e+06,714496.1664,1798,-0.0000,361.6167,744.3834,3
24,Baseline 1′ (idle_ratio 점수),같은 날 B1과 같은 개수,0.8738,4.075875e+06,713751.7830,1795,0.0000,0.0000,0.0000,0
30,Baseline 1′ (idle_ratio 점수),사후 맞춤 GPU-h (상한),0.8740,4.077391e+06,713751.7830,1795,0.0002,1515.2444,0.0000,0
36,Baseline 1′ (idle_ratio 점수),사후 맞춤 건수 (상한),0.8740,4.077391e+06,713751.7830,1795,0.0002,1515.2444,0.0000,0
42,Logistic Regression,F1 최대 (직전 주),0.8691,4.016799e+06,687299.8315,2487,-0.0047,-59076.7405,-26451.9515,692
48,Logistic Regression,위험 맞춤 GPU-h (직전 주),0.8678,3.998847e+06,678415.9543,2297,-0.0061,-77027.9402,-35335.8287,502
54,Logistic Regression,위험 맞춤 건수 (직전 주),0.8695,3.924000e+06,562370.8869,1796,-0.0043,-151875.8354,-151380.8962,1



=== E1 | 홀드아웃 5주 (147–180) | workload 전체 ===


,방법,정책,F1,회수 GPU-h,잘못 회수 GPU-h,잘못 회수 건수,ΔF1 vs B1,Δ회수 GPU-h vs B1,Δ잘못 회수 GPU-h vs B1,Δ잘못 회수 건수 vs B1
3,Baseline 1: 오늘 idle,-,0.8727,3.414183e+06,614490.5881,1453,0.0000,0.0000,0.0000,0
9,Baseline 1′ (idle_ratio 점수),F1 최대 (직전 주),0.8691,3.400639e+06,629065.5274,1459,-0.0036,-13543.3169,14574.9392,6
15,Baseline 1′ (idle_ratio 점수),위험 맞춤 GPU-h (직전 주),0.8727,3.414568e+06,615234.9715,1456,-0.0000,385.5889,744.3834,3
21,Baseline 1′ (idle_ratio 점수),위험 맞춤 건수 (직전 주),0.8727,3.414568e+06,615234.9715,1456,-0.0000,385.5889,744.3834,3
27,Baseline 1′ (idle_ratio 점수),같은 날 B1과 같은 개수,0.8727,3.414183e+06,614490.5881,1453,0.0000,0.0000,0.0000,0
33,Baseline 1′ (idle_ratio 점수),사후 맞춤 GPU-h (상한),0.8729,3.415698e+06,614490.5881,1453,0.0002,1515.2444,0.0000,0
39,Baseline 1′ (idle_ratio 점수),사후 맞춤 건수 (상한),0.8729,3.415698e+06,614490.5881,1453,0.0002,1515.2444,0.0000,0
45,Logistic Regression,F1 최대 (직전 주),0.8653,3.374014e+06,628763.7700,2136,-0.0074,-40168.6460,14273.1818,683
51,Logistic Regression,위험 맞춤 GPU-h (직전 주),0.8637,3.365187e+06,631803.3983,2009,-0.0090,-48996.0458,17312.8102,556
57,Logistic Regression,위험 맞춤 건수 (직전 주),0.8649,3.284005e+06,514049.9365,1482,-0.0078,-130177.1909,-100440.6517,29


In [11]:
# 주별 AP (점수 순서의 품질, threshold와 무관)
w1 = pd.DataFrame([r for r in WEEKLY if r["실험"] == "E1" and r["정책"] in ("-", "F1 최대 (직전 주)")])
display(w1.pivot_table(index="방법", columns="주", values="AP").round(4))
# 주별 Δ회수 (사후 맞춤 GPU-h, 같은 날 같은 개수) — 이득이 특정 주에 몰려 있는지
w2 = pd.DataFrame([r for r in WEEKLY if r["실험"] == "E1"])
b1w = w2[w2["방법"] == "Baseline 1: 오늘 idle"].set_index("주")
for pol in ["같은 날 B1과 같은 개수", "사후 맞춤 GPU-h (상한)", "위험 맞춤 GPU-h (직전 주)"]:
    t = w2[w2["정책"] == pol].copy()
    t["Δ회수 GPU-h"] = t["회수 GPU-h"] - t["주"].map(b1w["회수 GPU-h"])
    t["Δ잘못 회수 건수"] = t["잘못 회수 건수"] - t["주"].map(b1w["잘못 회수 건수"])
    print(f"\n주별 Δ vs B1 | 정책: {pol}")
    display(t.pivot_table(index="방법", columns="주", values=["Δ회수 GPU-h", "Δ잘못 회수 건수"]).round(0))

주,140–146,147–153,154–160,161–167,168–174,175–180
방법,,,,,,
Baseline 1: 오늘 idle,0.9084,0.7235,0.8581,0.9033,0.9329,0.9348
Baseline 1′ (idle_ratio 점수),0.9084,0.7235,0.8581,0.9033,0.9329,0.9348
LGBM+XGB 평균,0.9587,0.8878,0.9275,0.9629,0.9680,0.9705
LightGBM 튜닝,0.9586,0.8851,0.9276,0.9625,0.9684,0.9709
Logistic Regression,0.9398,0.8618,0.9074,0.9476,0.9601,0.9564
Random Forest,0.9412,0.8718,0.9211,0.9534,0.9710,0.9614
XGBoost 튜닝,0.9556,0.8859,0.9241,0.9607,0.9664,0.9686



주별 Δ vs B1 | 정책: 같은 날 B1과 같은 개수


Δ잘못 회수 건수                                         Δ회수 GPU-h                                            
주                             140–146 147–153 154–160 161–167 168–174 175–180   140–146  147–153  154–160  161–167  168–174 175–180
방법                                                                                                                                 
Baseline 1′ (idle_ratio 점수)       0.0     0.0     0.0     0.0     0.0     0.0       0.0      0.0      0.0      0.0      0.0     0.0
LGBM+XGB 평균                      -4.0    -6.0    -3.0    -3.0    -1.0    -3.0     -58.0   2324.0   -183.0  -1643.0   1220.0  1646.0
LightGBM 튜닝                       1.0   -14.0    -8.0    -7.0     0.0    -7.0   -1106.0  -5372.0  -1362.0  -3475.0   1490.0  2014.0
Logistic Regression              55.0    62.0    64.0    56.0    64.0    59.0  -13037.0 -12429.0 -13594.0 -20737.0 -15110.0 -6766.0
Random Forest                    -8.0   -14.0    -7.0    -6.0     4.0    -7.0   -5299.0  -1069.0   1726.0  -7508.0   9161.0  2062.0
XGBoost 튜닝                        0.0     2.0     1.0     0.0    -1.0    -1.0   -8271.0     93.0    -44.0  -1601.0    239.0   355.0


주별 Δ vs B1 | 정책: 사후 맞춤 GPU-h (상한)


Δ잘못 회수 건수                                         Δ회수 GPU-h                                            
주                             140–146 147–153 154–160 161–167 168–174 175–180   140–146  147–153  154–160  161–167  168–174 175–180
방법                                                                                                                                 
Baseline 1′ (idle_ratio 점수)       0.0     0.0     0.0     0.0     0.0     0.0       0.0     48.0     72.0    213.0   1182.0     0.0
LGBM+XGB 평균                       2.0    16.0    51.0   203.0    24.0    37.0    5554.0   5876.0   8396.0  24709.0   2349.0  4592.0
LightGBM 튜닝                      30.0     3.0    37.0   207.0    31.0    27.0    7905.0   7535.0   7075.0  22560.0   3227.0  5119.0
Logistic Regression             351.0    28.0    64.0   134.0    69.0    87.0   20138.0 -17209.0 -10487.0  -7274.0 -17588.0 -4498.0
Random Forest                   -28.0   -52.0   -31.0    16.0    -3.0   -11.0   -5892.0  -3235.0   5303.0   7224.0   1011.0  1748.0
XGBoost 튜닝                       14.0    36.0     4.0   120.0    11.0    37.0    7473.0   7933.0   3690.0  14614.0   2013.0  5425.0


주별 Δ vs B1 | 정책: 위험 맞춤 GPU-h (직전 주)


Δ잘못 회수 건수                                         Δ회수 GPU-h                                           
주                             140–146 147–153 154–160 161–167 168–174 175–180   140–146  147–153 154–160  161–167  168–174 175–180
방법                                                                                                                                
Baseline 1′ (idle_ratio 점수)       0.0     0.0     1.0     0.0     0.0     2.0     -24.0      0.0    72.0     50.0    240.0    24.0
LGBM+XGB 평균                      19.0    24.0   -10.0    10.0   178.0    64.0    6967.0   7044.0  2931.0   -889.0   8059.0  7608.0
LightGBM 튜닝                      30.0   -11.0    -7.0    19.0   149.0    43.0    7905.0   5148.0  4295.0   2489.0   8888.0  5902.0
Logistic Regression             -54.0   115.0   205.0    67.0    68.0   101.0  -28032.0  -1855.0  1273.0 -27386.0 -17612.0 -3415.0
Random Forest                   -27.0   -10.0   -31.0   -14.0    10.0     3.0   -5892.0  10534.0  5303.0  -6795.0   1587.0  3175.0
XGBoost 튜닝                       -6.0    -1.0     9.0     3.0    93.0    45.0    2873.0   4766.0  4235.0  -4613.0   4697.0  5449.0

In [12]:
# E1 부트스트랩 (app 단위) — 운영에서 쓸 수 있는 두 정책
boots = []
for pool, names in POOLS.items():
    idx = [i for i, W in enumerate(WK) if W["name"] in names]
    pool_df = pd.concat([WK[i]["wk"] for i in idx])
    base = np.concatenate([WK[i]["b1k"] for i in idx])
    for pol in ["위험 맞춤 GPU-h (직전 주)", "같은 날 B1과 같은 개수", "사후 맞춤 GPU-h (상한)"]:
        preds = {m: np.concatenate([PRED_STORE[("E1", "기본", m, 0, pol)][i] for i in idx])
                 for m in ["Baseline 1′ (idle_ratio 점수)", *E1_MODELS, "LGBM+XGB 평균"]}
        bt = boot_diff(pool_df, preds, base)
        bt.insert(0, "정책", pol); bt.insert(0, "구간", pool)
        boots.append(bt)
boots = pd.concat(boots)
boots.to_csv(OUT / "e1_bootstrap.csv", index=False)
display(boots)
print("누적", elapsed())

,구간,정책,방법,Δ회수 GPU-h,Δ잘못 회수 GPU-h,Δ잘못 회수 건수,ΔF1
0,6주 (140–180),위험 맞춤 GPU-h (직전 주),Baseline 1′ (idle_ratio 점수),"+376.7 [+24, +958.5]","+762.3 [+0, +2,160]","+3.026 [+0, +7]","-2.572e-05 [-0.0001806, +9.27e-05]"
1,6주 (140–180),위험 맞춤 GPU-h (직전 주),Logistic Regression,"-7.763e+04 [-1.199e+05, -3.935e+04]","-3.568e+04 [-1.209e+05, +3.925e+04]","+499.7 [+366, +649]","-0.006175 [-0.01425, +0.00249]"
2,6주 (140–180),위험 맞춤 GPU-h (직전 주),Random Forest,"+7,299 [-1.756e+04, +3.086e+04]","-1.182e+04 [-1.045e+05, +7.127e+04]","-69.07 [-104, -34]","+0.001988 [-0.006259, +0.01118]"
3,6주 (140–180),위험 맞춤 GPU-h (직전 주),LightGBM 튜닝,"+3.407e+04 [+1.174e+04, +6.146e+04]","-2.167e+04 [-1.066e+05, +4.427e+04]","+223.1 [+166, +276]","+0.006143 [-0.0009235, +0.0141]"
4,6주 (140–180),위험 맞춤 GPU-h (직전 주),XGBoost 튜닝,"+1.718e+04 [+1,245, +3.656e+04]","-4.583e+04 [-1.441e+05, +3.623e+04]","+142.5 [+104, +180]","+0.006389 [-0.001779, +0.01581]"
5,6주 (140–180),위험 맞춤 GPU-h (직전 주),LGBM+XGB 평균,"+3.141e+04 [+1.518e+04, +5.198e+04]","-9,255 [-9.92e+04, +6.436e+04]","+284.4 [+233, +333]","+0.004651 [-0.002636, +0.01325]"
0,6주 (140–180),같은 날 B1과 같은 개수,Baseline 1′ (idle_ratio 점수),"+0 [+0, +0]","+0 [+0, +0]","+0 [+0, +0]","+0 [+0, +0]"
1,6주 (140–180),같은 날 B1과 같은 개수,Logistic Regression,"-8.22e+04 [-1.193e+05, -4.804e+04]","-4.283e+04 [-1.31e+05, +3.622e+04]","+359.2 [+241, +491]","-0.006069 [-0.01459, +0.002435]"
2,6주 (140–180),같은 날 B1과 같은 개수,Random Forest,"-1,267 [-3.475e+04, +3.245e+04]","+6,946 [-5.639e+04, +6.307e+04]","-37.89 [-71, -5]","-0.0008113 [-0.007009, +0.006147]"
3,6주 (140–180),같은 날 B1과 같은 개수,LightGBM 튜닝,"-8,294 [-3.682e+04, +1.841e+04]","-6.106e+04 [-1.423e+05, +4,639]","-34.63 [-66, -6]","+0.004748 [-0.002073, +0.01282]"


누적 26.1분


In [13]:
# E1 workload별 (홀드아웃 5주) — 이득/손해가 online·offline 어디서 나는지
e1w = pd.concat([pooled_view("E1", seg="online"), pooled_view("E1", seg="offline")])
e1w.to_csv(OUT / "e1_pooled_by_workload.csv", index=False)
pool = list(POOLS)[-1]
display(e1w[(e1w["구간"] == pool) & e1w["정책"].isin(["-", "같은 날 B1과 같은 개수", "사후 맞춤 GPU-h (상한)", "위험 맞춤 GPU-h (직전 주)"])]
        [["workload", "방법", "정책", "F1", "Δ회수 GPU-h vs B1", "Δ잘못 회수 GPU-h vs B1", "Δ잘못 회수 건수 vs B1"]].round(4))

,workload,방법,정책,F1,Δ회수 GPU-h vs B1,Δ잘못 회수 GPU-h vs B1,Δ잘못 회수 건수 vs B1
4,online,Baseline 1: 오늘 idle,-,0.8997,0.0000,0.0000,0
16,online,Baseline 1′ (idle_ratio 점수),위험 맞춤 GPU-h (직전 주),0.8997,385.5889,744.3834,3
28,online,Baseline 1′ (idle_ratio 점수),같은 날 B1과 같은 개수,0.8997,0.0000,0.0000,0
34,online,Baseline 1′ (idle_ratio 점수),사후 맞춤 GPU-h (상한),0.9000,1515.2444,0.0000,0
52,online,Logistic Regression,위험 맞춤 GPU-h (직전 주),0.8860,-50828.2457,25445.2329,520
64,online,Logistic Regression,같은 날 B1과 같은 개수,0.8879,-63561.5454,-2806.2337,290
70,online,Logistic Regression,사후 맞춤 GPU-h (상한),0.8902,-49765.5452,-57.1724,360
88,online,Random Forest,위험 맞춤 GPU-h (직전 주),0.9002,11916.8558,11383.5780,-27
100,online,Random Forest,같은 날 B1과 같은 개수,0.8997,8311.7670,10124.0612,-15
106,online,Random Forest,사후 맞춤 GPU-h (상한),0.9026,17071.1891,1995.4117,-62


## 5. E2 — seed에 따른 흔들림

LightGBM·XGBoost 튜닝 설정(subsample·colsample이 1보다 작아 난수 영향이 있음)을 seed 0–4로 다시 학습.
seed마다 합계 Δ vs Baseline 1을 내고 평균·표준편차·최소·최대를 본다.

In [14]:
SEEDS = [0, 1] if SMOKE else [0, 1, 2, 3, 4]
for name, kind in [("LightGBM 튜닝", "lgbm"), ("XGBoost 튜닝", "xgb")]:
    for sd in SEEDS:
        sc = E1_SCORES[name] if sd == 0 else walk_forward(kind, BEST[kind], seed=sd)
        record("E2", "seed", name, sd, week_scores=sc)
    print(f"E2 {name}: 누적 {elapsed()}", flush=True)
e2 = pooled_view("E2")
e2.to_csv(OUT / "e2_seeds_pooled.csv", index=False)
dcols = ["ΔF1 vs B1", "Δ회수 GPU-h vs B1", "Δ잘못 회수 GPU-h vs B1", "Δ잘못 회수 건수 vs B1"]
e2s = e2.groupby(["구간", "방법", "정책"])[dcols].agg(["mean", "std", "min", "max"])
e2s.to_csv(OUT / "e2_seeds_summary.csv")
display(e2s.round(3))

E2 LightGBM 튜닝: 누적 62.3분


E2 XGBoost 튜닝: 누적 66.3분


ΔF1 vs B1                      Δ회수 GPU-h vs B1                                   Δ잘못 회수 GPU-h vs B1                                     \
                                                      mean    std    min    max            mean        std         min        max               mean        std         min         max   
구간                방법          정책                                                                                                                                                          
6주 (140–180)      LightGBM 튜닝 F1 최대 (직전 주)           0.005  0.004  0.001  0.011      -90691.029  18921.860 -114886.856 -72261.423        -165047.415  33728.158 -216001.482 -126974.518   
                              같은 날 B1과 같은 개수         0.005  0.001  0.005  0.006      -11794.430   7407.074  -23022.939  -3589.650         -70483.995   6200.939  -77870.862  -60991.151   
                              사후 맞춤 GPU-h (상한)       0.006  0.002  0.004  0.009       41963.229   7812.051   33802.550  53420.150         -11036.225  14660.080  -35941.479   -1142.635   
                              사후 맞춤 건수 (상한)          0.007  0.002  0.006  0.010        2906.563   5702.300   -6394.500   7827.867         -68763.965  14593.367  -93900.007  -56434.351   
                              위험 맞춤 GPU-h (직전 주)     0.005  0.001  0.003  0.006       31367.374   5015.864   26007.494  37234.956         -16292.949  11138.127  -28221.378     107.127   
                              위험 맞춤 건수 (직전 주)        0.007  0.001  0.006  0.009        7616.201   8575.837   -3223.567  16715.639         -64223.914  17464.683  -93022.295  -50331.029   
                  XGBoost 튜닝  F1 최대 (직전 주)           0.004  0.001  0.002  0.006      -76982.826  18742.115 -107106.501 -59588.917        -142717.227  12303.293 -157433.263 -126729.035   
                              같은 날 B1과 같은 개수         0.007  0.000  0.006  0.007       -5691.543   2548.242   -9228.605  -3164.122         -78035.554   3563.321  -82618.296  -73174.151   
                              사후 맞춤 GPU-h (상한)       0.007  0.002  0.005  0.009       38436.618   9422.668   23210.639  48340.378         -20221.743  12988.125  -37487.601   -4143.379   
                              사후 맞춤 건수 (상한)          0.009  0.002  0.007  0.012        8223.722   8253.233    3268.645  22698.811         -83129.643  10293.373  -94341.862  -71846.829   
                              위험 맞춤 GPU-h (직전 주)     0.006  0.003  0.001  0.010       17639.796   5021.003   11680.395  25150.344         -39625.621  31024.219  -71469.734    4575.788   
                              위험 맞춤 건수 (직전 주)        0.008  0.001  0.007  0.010        9707.461   6838.580    3887.967  20820.078         -74455.897  11636.647  -90271.412  -61872.990   
홀드아웃 5주 (147–180) LightGBM 튜닝 F1 최대 (직전 주)           0.004  0.004  0.001  0.010      -93581.798  16419.702 -111312.717 -74903.317        -155087.538  24600.418 -192226.793 -124038.790   
                              같은 날 B1과 같은 개수         0.007  0.001  0.006  0.007       -8364.514   4523.183  -15946.889  -4571.483         -69360.620   5232.105  -74503.890  -60522.284   
                              사후 맞춤 GPU-h (상한)       0.006  0.001  0.004  0.007       34836.060   7129.697   27891.589  45515.289          -6598.849   7130.795  -16214.846   -1094.423   
                              사후 맞춤 건수 (상한)          0.007  0.001  0.006  0.008         156.791   3629.898   -4654.878   4793.933         -61690.360   5041.948  -69754.885  -55853.690   
                              위험 맞춤 GPU-h (직전 주)     0.005  0.003  0.001  0.007       24549.423   5088.589   16986.633  30962.772         -11533.767  22440.043  -29852.734   26288.083   
                              위험 맞춤 건수 (직전 주)        0.007  0.001  0.007  0.008        6954.641   5528.693     278.567  12960.055         -56654.133   6274.669  -64548.407  -50853.229   
                  XGBoost 튜닝  F1 최대 (직전 주)           0.003  0.002  0.001  0.005      -79592.749  23158.035 -117691.851 -62

## 6. E3 — 변수 묶음 (LightGBM 튜닝, seed 0)

v2 입력 변수 23개를 6묶음으로 나눈다. 묶음을 "뺄" 때는 그 열을 상수로 바꾼다(트리가 쓰지 못함, 열 구성·인코더는 그대로).
- **더하기**: idle_ratio만 → +이력 → +오늘 사용량 → +규모 → +메타 → +달력(= 전체)
- **하나씩 빼기**: 전체에서 한 묶음만 뺌

In [15]:
GROUPS = {
    "오늘 idle": ["idle_ratio"],
    "이력": ["idle_ratio_lag1", "idle_streak"],
    "오늘 사용량": ["sm_util", "gpu_mem_util", "mem_usage", "outlier_sm_util", "outlier_gpu_mem_util", "outlier_mem_usage"],
    "규모": ["gpu_type", "gpu_count", "gpu_mem_alloc", "gpu_hours", "gpu_hours_per_gpu"],
    "메타": ["workload", "pd_role", "params_b", "model_arch", "base_arch", "variant_arch", "cluster_id", "site_id"],
    "달력": ["day_of_week"],
}
X_all = split_xyw(df.head())[0].columns
assert sorted(sum(GROUPS.values(), [])) == sorted(X_all), set(X_all) ^ set(sum(GROUPS.values(), []))


def ablate(keep_groups):
    d = df.copy()
    for g, cols in GROUPS.items():
        if g in keep_groups:
            continue
        for c in cols:
            if isinstance(d[c].dtype, pd.CategoricalDtype):
                d[c] = pd.Categorical([d[c].cat.categories[0]] * len(d), categories=d[c].cat.categories)
            else:
                d[c] = np.zeros(len(d), dtype=d[c].dtype)
    return d


order = list(GROUPS)
configs = {}
for i in range(1, len(order)):          # 더하기 (전체는 E1 결과 재사용)
    configs["더하기: " + " + ".join(order[:i])] = order[:i]
for g in order[1:]:                       # 하나씩 빼기 (오늘 idle 빼기도 포함)
    configs[f"빼기: −{g}"] = [x for x in order if x != g]
configs["빼기: −오늘 idle"] = order[1:]
if SMOKE:
    configs = dict(list(configs.items())[:2])
record("E3", "전체 (23개)", "LightGBM 튜닝", 0, week_scores=E1_SCORES["LightGBM 튜닝"])
for cname, keep in configs.items():
    record("E3", cname, "LightGBM 튜닝", 0, week_scores=walk_forward("lgbm", BEST["lgbm"], data=ablate(keep)))
    print(f"E3 {cname}: 누적 {elapsed()}", flush=True)
e3 = pooled_view("E3", ["같은 날 B1과 같은 개수", "사후 맞춤 GPU-h (상한)", "위험 맞춤 GPU-h (직전 주)"])
e3w = pd.DataFrame([r for r in WEEKLY if r["실험"] == "E3" and r["정책"] == "F1 최대 (직전 주)"])
ap = e3w.groupby("설정")["AP"].mean().rename("주별 AP 평균")
e3 = e3.merge(ap, left_on="설정", right_index=True, how="left")
e3.to_csv(OUT / "e3_ablation.csv", index=False)
for pool in POOLS:
    print(f"\n=== E3 | {pool} ===")
    display(e3[e3["구간"] == pool].pivot_table(index="설정", columns="정책", values="Δ회수 GPU-h vs B1", sort=False).round(0)
            .join(e3[e3["구간"] == pool].groupby("설정")["주별 AP 평균"].first()))

E3 더하기: 오늘 idle: 누적 74.5분


E3 더하기: 오늘 idle + 이력: 누적 88.9분


E3 더하기: 오늘 idle + 이력 + 오늘 사용량: 누적 95.9분


E3 더하기: 오늘 idle + 이력 + 오늘 사용량 + 규모: 누적 113.2분


E3 더하기: 오늘 idle + 이력 + 오늘 사용량 + 규모 + 메타: 누적 120.5분


E3 빼기: −이력: 누적 126.5분


... (5 outputs left out)


=== E3 | 6주 (140–180) ===


,위험 맞춤 GPU-h (직전 주),같은 날 B1과 같은 개수,사후 맞춤 GPU-h (상한),주별 AP 평균
설정,,,,
전체 (23개),34626.0,-7811.0,53420.0,0.945523
더하기: 오늘 idle,-42321.0,-840.0,-27420.0,0.847130
더하기: 오늘 idle + 이력,-27059.0,-5203.0,-16212.0,0.928664
더하기: 오늘 idle + 이력 + 오늘 사용량,12816.0,3197.0,18351.0,0.936073
더하기: 오늘 idle + 이력 + 오늘 사용량 + 규모,11552.0,-12144.0,38066.0,0.937761
더하기: 오늘 idle + 이력 + 오늘 사용량 + 규모 + 메타,2949.0,-15781.0,39785.0,0.943375
빼기: −이력,12985.0,-30544.0,34392.0,0.938150
빼기: −오늘 사용량,27616.0,-12905.0,46621.0,0.944753
빼기: −규모,20804.0,21499.0,26868.0,0.945860



=== E3 | 홀드아웃 5주 (147–180) ===


,위험 맞춤 GPU-h (직전 주),같은 날 B1과 같은 개수,사후 맞춤 GPU-h (상한),주별 AP 평균
설정,,,,
전체 (23개),26721.0,-6704.0,45515.0,0.945523
더하기: 오늘 idle,-37508.0,-816.0,-27396.0,0.847130
더하기: 오늘 idle + 이력,-26668.0,-4014.0,-23690.0,0.928664
더하기: 오늘 idle + 이력 + 오늘 사용량,3182.0,-3534.0,2756.0,0.936073
더하기: 오늘 idle + 이력 + 오늘 사용량 + 규모,7132.0,-11109.0,25747.0,0.937761
더하기: 오늘 idle + 이력 + 오늘 사용량 + 규모 + 메타,-7671.0,-12839.0,29458.0,0.943375
빼기: −이력,4973.0,-34263.0,20306.0,0.938150
빼기: −오늘 사용량,22147.0,-6501.0,42691.0,0.944753
빼기: −규모,17567.0,20417.0,19238.0,0.945860


## 7. E4 — 학습 기간 (LightGBM·XGBoost 튜닝, seed 0)

매주 학습할 때 전체 과거 / 최근 56일 / 최근 28일만 쓴다. 최근 기간이 낫다면 분포 변화(drift)가 홀드아웃 성능을 깎는 것이다.

In [16]:
WINDOWS = [None, 28] if SMOKE else [None, 56, 28]
for name, kind in [("LightGBM 튜닝", "lgbm"), ("XGBoost 튜닝", "xgb")]:
    for win in WINDOWS:
        lab = "전체 과거" if win is None else f"최근 {win}일"
        sc = E1_SCORES[name] if win is None else walk_forward(kind, BEST[kind], window=win)
        record("E4", lab, name, 0, week_scores=sc)
        print(f"E4 {name} {lab}: 누적 {elapsed()}", flush=True)
e4 = pooled_view("E4", ["같은 날 B1과 같은 개수", "사후 맞춤 GPU-h (상한)", "위험 맞춤 GPU-h (직전 주)"])
e4w = pd.DataFrame([r for r in WEEKLY if r["실험"] == "E4" and r["정책"] == "F1 최대 (직전 주)"])
e4 = e4.merge(e4w.groupby(["설정", "방법"])["AP"].mean().rename("주별 AP 평균").reset_index(), on=["설정", "방법"], how="left")
e4.to_csv(OUT / "e4_window.csv", index=False)
for pool in POOLS:
    print(f"\n=== E4 | {pool} ===")
    display(e4[e4["구간"] == pool][["방법", "설정", "정책", "주별 AP 평균", "F1", "Δ회수 GPU-h vs B1", "Δ잘못 회수 GPU-h vs B1",
                                    "Δ잘못 회수 건수 vs B1"]].round(4))

E4 LightGBM 튜닝 전체 과거: 누적 181.7분


E4 LightGBM 튜닝 최근 56일: 누적 189.0분


E4 LightGBM 튜닝 최근 28일: 누적 192.6분


E4 XGBoost 튜닝 전체 과거: 누적 192.6분


E4 XGBoost 튜닝 최근 56일: 누적 192.9분


E4 XGBoost 튜닝 최근 28일: 누적 193.2분



=== E4 | 6주 (140–180) ===


,방법,설정,정책,주별 AP 평균,F1,Δ회수 GPU-h vs B1,Δ잘못 회수 GPU-h vs B1,Δ잘못 회수 건수 vs B1
0,LightGBM 튜닝,전체 과거,위험 맞춤 GPU-h (직전 주),0.9455,0.8800,34626.1000,-21437.6506,223
2,LightGBM 튜닝,전체 과거,같은 날 B1과 같은 개수,0.9455,0.8786,-7810.7500,-60991.1507,-35
4,LightGBM 튜닝,전체 과거,사후 맞춤 GPU-h (상한),0.9455,0.8803,53420.1502,-1142.6347,335
6,LightGBM 튜닝,최근 56일,위험 맞춤 GPU-h (직전 주),0.9475,0.8788,26184.7222,-18972.5115,379
8,LightGBM 튜닝,최근 56일,같은 날 B1과 같은 개수,0.9475,0.8814,-24999.7499,-111866.4074,-30
10,LightGBM 튜닝,최근 56일,사후 맞춤 GPU-h (상한),0.9475,0.8794,41486.9945,-6297.3181,388
12,LightGBM 튜닝,최근 28일,위험 맞춤 GPU-h (직전 주),0.9458,0.8799,30169.4278,-26482.9410,598
14,LightGBM 튜닝,최근 28일,같은 날 B1과 같은 개수,0.9458,0.8785,-40590.6337,-101848.4295,1
16,LightGBM 튜닝,최근 28일,사후 맞춤 GPU-h (상한),0.9458,0.8830,52209.8001,-30762.6860,697
18,XGBoost 튜닝,전체 과거,위험 맞춤 GPU-h (직전 주),0.9436,0.8801,17406.8945,-44974.3786,143



=== E4 | 홀드아웃 5주 (147–180) ===


,방법,설정,정책,주별 AP 평균,F1,Δ회수 GPU-h vs B1,Δ잘못 회수 GPU-h vs B1,Δ잘못 회수 건수 vs B1
1,LightGBM 튜닝,전체 과거,위험 맞춤 GPU-h (직전 주),0.9455,0.8789,26721.2389,-21423.5505,193
3,LightGBM 튜닝,전체 과거,같은 날 B1과 같은 개수,0.9455,0.8785,-6704.3333,-60522.2840,-36
5,LightGBM 튜닝,전체 과거,사후 맞춤 GPU-h (상한),0.9455,0.8793,45515.2891,-1128.5346,305
7,LightGBM 튜닝,최근 56일,위험 맞춤 GPU-h (직전 주),0.9475,0.8779,26598.7333,-12289.9836,346
9,LightGBM 튜닝,최근 56일,같은 날 B1과 같은 개수,0.9475,0.8827,-14329.1943,-106806.7406,-22
11,LightGBM 튜닝,최근 56일,사후 맞춤 GPU-h (상한),0.9475,0.8783,37832.8279,-1783.4180,335
13,LightGBM 튜닝,최근 28일,위험 맞춤 GPU-h (직전 주),0.9458,0.8800,37056.4000,-18057.9242,584
15,LightGBM 튜닝,최근 28일,같은 날 B1과 같은 개수,0.9458,0.8786,-28043.9948,-88679.9405,1
17,LightGBM 튜닝,최근 28일,사후 맞춤 GPU-h (상한),0.9458,0.8830,48122.8723,-30731.8582,610
19,XGBoost 튜닝,전체 과거,위험 맞춤 GPU-h (직전 주),0.9436,0.8773,14533.5056,-22947.0621,149


## 8. 저장

In [17]:
pd.DataFrame(WEEKLY).to_csv(OUT / "weekly_all.csv", index=False)
pd.DataFrame(POOLED).to_csv(OUT / "pooled_all.csv", index=False)
# 점수 저장: 주별 행 순서는 df에서 그 주를 고른 순서 그대로
key_rows = []
for W in WK:
    key_rows.append(pd.DataFrame({"주": W["name"], "row_index": W["wk"].index, "app_id": W["wk"]["app_id"].astype(str).values,
                                  "day": W["day"], "y": W["yk"], "w": W["wk_w"], "b1": W["b1k"], "idle_ratio": W["s1k"]}))
keys = pd.concat(key_rows, ignore_index=True)
cols = {}
for (exp, setting, method, seed, wname) in SCORE_STORE:
    cols.setdefault(f"{exp}|{setting}|{method}|seed{seed}", None)
for cname in cols:
    exp, setting, method, seed = cname.split("|")
    cols[cname] = np.concatenate([SCORE_STORE[(exp, setting, method, int(seed[4:]), W["name"])] for W in WK]).astype("float32")
scores = pd.concat([keys, pd.DataFrame(cols)], axis=1)
scores.to_parquet(OUT / "scores.parquet", index=False)
print(f"저장: {OUT} | 점수 {scores.shape} | 전체 {elapsed()}")

저장: /ssd/jylee/gpu-reclaim/outputs/v4 | 점수 (121183, 42) | 전체 193.2분
